# Gravity / Sun, Earth & Luna

A moving wireframe landscape: Earth orbits on the Sun's broad well, and Luna
orbits on the combined Sun-and-Earth surface. Each body adds its own depression.

**Run all cells** with the project's `.venv` kernel to regenerate
`output/20261018_gravity.html` and the interactive preview below. Serve the repository
with `python -m http.server 8000` to open the exported page. All geometry, procedural
textures, browser callbacks, and page layout are defined in this notebook.

### What this picture means

General relativity describes gravity through spacetime geometry; freely falling test
bodies follow geodesics. This is a **3D schematic using potential-shaped wells**,
not a coordinate projection of a computed spacetime metric. Height is a drawing
coordinate, not a physical downward direction or the time dimension. The orbits are
prescribed circular guides, not trajectories computed by rolling balls or solving
Einstein's equations. Real gravity has no rim or finite outer edge.

Body sizes, the Earth–Luna separation, vertical scale, and the default Earth/Luna
contributions are enlarged for visibility. **Real mass ratio** removes the common
amplification of Earth and Luna's wells; the other drawing conventions remain.
Depth changes drawing heights, not physical masses or orbital periods.

References (accessed 18 October 2026):
- [NASA Earth fact sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/earthfact.html)
- [NASA Sun fact sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/sunfact.html)
- [NASA Moon fact sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/moonfact.html)
- [Einstein Online: geometric gravity](https://www.einstein-online.info/en/GeomGravity/)


In [ ]:
from pathlib import Path
from html import escape
import json
import sys

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display

current = Path.cwd().resolve()
project_root = next((p for p in (current, *current.parents)
                     if (p / "site_export.py").is_file() and (p / "notebooks").is_dir()), None)
if project_root is None:
    raise RuntimeError("Run from the repository root or the notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from site_export import write_plot_page, PLOTLY_URL

# NASA masses in kg; time and distance are used for the circular-orbit guide.
SUN_MASS = 1.9884e30
EARTH_MASS = 5.9722e24
MOON_MASS = 7.346e22
MOON_MASS_RATIO = MOON_MASS / SUN_MASS
LUNAR_DAYS = 27.3217
MASS_RATIO = EARTH_MASS / SUN_MASS
YEAR_DAYS = 365.256
ORBIT_RADIUS = 1.0  # AU; drawing coordinates x and y are in AU.

# Drawing parameters, not measurements of physical curvature or body radii.
EXTENT = 1.65
# Display falloff radii: wells return smoothly to the reference plane.
# These are drawing parameters, not finite physical ranges of gravity.
SUN_WELL_RADIUS = 2.2
EARTH_WELL_RADIUS = 0.45
MOON_WELL_RADIUS = 0.13
MOON_ORBIT_RADIUS = 0.22  # Exaggerated: the real mean distance is 384,400 km.
Z_RANGE = (-2.65, 0.35)
# Match scene-box proportions to fixed axis spans: one unit has the same
# displayed length on every axis, independent of the animated data extents.
SCENE_ASPECT = dict(x=1, y=1, z=(Z_RANGE[1] - Z_RANGE[0]) / (2 * EXTENT))
SUN_SOFTENING = 0.17
EARTH_SOFTENING = 0.07
MOON_SOFTENING = 0.022
POTENTIAL_SCALE = 0.25
SUN_RADIUS = 0.22
EARTH_RADIUS = 0.085
MOON_RADIUS = 0.034  # Enlarged independently so Luna stays visible.
DEFAULT_GAIN = 20_000
DEFAULT_DEPTH = 1.0
INITIAL_PHASE = 320.0
SECONDS_PER_YEAR = 24.0
BACKGROUND = "#04080e"
CAMERA = dict(eye=dict(x=1.45, y=-1.8, z=1.55), up=dict(x=0, y=0, z=1),
              projection=dict(type="perspective"))

## 1. Orbits follow the parent surface

Earth's center and blue orbit guide use the **Sun-only** height. Luna's center and
silver guide use the **Sun + Earth** height. A body's own contribution deepens the
grid beneath it; it does not pull its own center downward. The Sun stays at the
reference height, centered over its own depression. This is an explicit illustration
convention; it does not impose physical contact or simulate rolling.

For body $i$, the drawing contribution is

$$h_i(x,y)=-ADq_i f_i\frac{\max(0,1-r_i^2/R_i^2)^3}
{\sqrt{r_i^2+\epsilon_i^2}},\qquad z_{grid}=h_\odot+h_\oplus+h_{Luna}.$$

Here $q_i=M_i/M_\odot$, $f_\odot=1$, and Earth and Luna share the selected gain
$f$. Distances $r_i$ are horizontal distances from body centers. The smooth taper
is a display device, not a physical cutoff of gravity. The Sun's display radius is
2.2, so the entire 1 AU Earth orbit lies **inside the sloping solar well**. The lunar
orbit has drawing radius 0.22, inside Earth's display radius 0.45. Its height varies
around Earth because the solar background is sloped.

As Earth moves away, its dip disappears from that point, leaving the Sun's surface.
Likewise Luna leaves the Sun-and-Earth surface behind. Changing depth moves both the
grid and the orbit heights together. At zero depth all centers and guides lie at $z=0$.

The circular, coplanar guides use periods of 365.256 and 27.3217 days. Luna's real orbit
is eccentric and inclined; both that complexity and Earth–Moon barycentric motion
are omitted. The initial phase is arbitrary, not an ephemeris. Continuous elapsed
time preserves lunar motion across Earth-year boundaries; scrubbing selects a day
in the first model year. The enlarged lunar separation and sphere sizes are not to scale.


In [ ]:
def earth_position(phase_degrees):
    angle = phase_degrees * np.pi / 180
    return ORBIT_RADIUS*np.cos(angle), ORBIT_RADIUS*np.sin(angle)


def moon_position(phase_degrees):
    ex, ey = earth_position(phase_degrees)
    angle = phase_degrees * (YEAR_DAYS/LUNAR_DAYS) * np.pi / 180
    return ex+MOON_ORBIT_RADIUS*np.cos(angle), ey+MOON_ORBIT_RADIUS*np.sin(angle)


def localized_well(distance_squared, softening, display_radius):
    """Potential-shaped profile with a C2-smooth return to the reference plane."""
    distance_squared = np.asarray(distance_squared)
    weight = np.maximum(0, 1-distance_squared/display_radius**2)**3
    return weight/np.sqrt(distance_squared+softening**2)


def solar_height(x, y, depth=DEFAULT_DEPTH):
    x, y = np.asarray(x), np.asarray(y)
    return -POTENTIAL_SCALE*depth*localized_well(x*x+y*y, SUN_SOFTENING, SUN_WELL_RADIUS)


def parent_height(x, y, phase, depth=DEFAULT_DEPTH, gain=DEFAULT_GAIN):
    """Surface beneath Luna's orbit: Sun and Earth, excluding Luna itself."""
    x, y = np.asarray(x), np.asarray(y)
    ex, ey = earth_position(phase)
    earth = MASS_RATIO*gain*localized_well((x-ex)**2+(y-ey)**2, EARTH_SOFTENING, EARTH_WELL_RADIUS)
    return solar_height(x, y, depth)-POTENTIAL_SCALE*depth*earth


def grid_height(x, y, phase, depth=DEFAULT_DEPTH, gain=DEFAULT_GAIN):
    x, y = np.asarray(x), np.asarray(y)
    mx, my = moon_position(phase)
    moon = MOON_MASS_RATIO*gain*localized_well((x-mx)**2+(y-my)**2, MOON_SOFTENING, MOON_WELL_RADIUS)
    return parent_height(x, y, phase, depth, gain)-POTENTIAL_SCALE*depth*moon


def body_centers(phase, depth=DEFAULT_DEPTH, gain=DEFAULT_GAIN):
    ex, ey = earth_position(phase)
    mx, my = moon_position(phase)
    return [(0., 0., 0., SUN_RADIUS),
            (float(ex), float(ey), float(solar_height(ex, ey, depth)), EARTH_RADIUS),
            (float(mx), float(my), float(parent_height(mx, my, phase, depth, gain)), MOON_RADIUS)]


# Full lines continue into the depressions; no equatorial holes or forced joins.
grid_x, grid_y = [], []
samples = np.linspace(-EXTENT, EXTENT, 161).tolist()
for fixed in np.linspace(-EXTENT, EXTENT, 51):
    for axis in (0, 1):
        grid_x.extend(samples if axis == 0 else [float(fixed)]*len(samples))
        grid_y.extend([float(fixed)]*len(samples) if axis == 0 else samples)
        grid_x.append(None); grid_y.append(None)

orbit_angles = np.linspace(0, 2*np.pi, 181)
orbit_x = ORBIT_RADIUS*np.cos(orbit_angles)
orbit_y = ORBIT_RADIUS*np.sin(orbit_angles)
moon_offset_x = MOON_ORBIT_RADIUS*np.cos(orbit_angles)
moon_offset_y = MOON_ORBIT_RADIUS*np.sin(orbit_angles)
longitude, latitude = np.meshgrid(np.linspace(-np.pi, np.pi, 65), np.linspace(-np.pi/2, np.pi/2, 33))
unit_sphere = (np.cos(latitude)*np.cos(longitude), np.cos(latitude)*np.sin(longitude), np.sin(latitude))


def sphere(center, radius):
    return [(center[axis]+radius*unit_sphere[axis]).tolist() for axis in range(3)]


def scene_geometry(phase=INITIAL_PHASE, depth=DEFAULT_DEPTH, gain=DEFAULT_GAIN):
    centers = body_centers(phase, depth, gain)
    sun, earth, moon = centers
    indices = [i for i, x in enumerate(grid_x) if x is not None]
    heights = grid_height(np.array([grid_x[i] for i in indices]), np.array([grid_y[i] for i in indices]), phase, depth, gain)
    grid_z = [None]*len(grid_x)
    for i, height in zip(indices, heights):
        grid_z[i] = float(height)
    sun_xyz, earth_xyz, moon_xyz = [sphere(c[:3], c[3]) for c in centers]
    lunar_x, lunar_y = earth[0]+moon_offset_x, earth[1]+moon_offset_y
    return dict(
        x=[grid_x, orbit_x.tolist(), sun_xyz[0], earth_xyz[0], [c[0] for c in centers], moon_xyz[0], lunar_x.tolist()],
        y=[grid_y, orbit_y.tolist(), sun_xyz[1], earth_xyz[1], [c[1] for c in centers], moon_xyz[1], lunar_y.tolist()],
        z=[grid_z, solar_height(orbit_x, orbit_y, depth).tolist(), sun_xyz[2], earth_xyz[2],
           [c[2]+c[3]+offset for c, offset in zip(centers, [0.08, 0.07, 0.045])], moon_xyz[2],
           parent_height(lunar_x, lunar_y, phase, depth, gain).tolist()],
    )


# Geometry invariants: parent surfaces, self-wells, roundness, and continuous time.
assert 3.0e-6 < MASS_RATIO < 3.01e-6
assert SUN_WELL_RADIUS > ORBIT_RADIUS+MOON_ORBIT_RADIUS
assert EARTH_WELL_RADIUS > MOON_ORBIT_RADIUS
display_units = np.array(list(SCENE_ASPECT.values())) / np.array([2*EXTENT, 2*EXTENT, Z_RANGE[1]-Z_RANGE[0]])
assert np.allclose(display_units, display_units[0])
assert solar_height(0.99, 0) < solar_height(1.01, 0) < 0  # Orbit is on a slope.
for phase in np.linspace(0, 720, 49):
    for depth, gain in [(0, 1), (1, 20_000), (1.6, 50_000)]:
        centers = body_centers(phase, depth, gain)
        sun, earth, moon = centers
        assert np.isclose(earth[2], solar_height(*earth[:2], depth))
        assert np.isclose(moon[2], parent_height(*moon[:2], phase, depth, gain))
        assert np.isclose(np.linalg.norm(np.array(moon[:2])-earth[:2]), MOON_ORBIT_RADIUS)
        if depth:
            assert grid_height(*earth[:2], phase, depth, gain) < earth[2] < 0
            assert grid_height(*moon[:2], phase, depth, gain) < moon[2]
        # A vacated Earth position returns to the solar surface, not to a flat plane.
        assert np.isclose(grid_height(*earth[:2], phase+90, depth, gain), earth[2])
        assert grid_height(EXTENT, EXTENT, phase, depth, gain) == 0
for phase, depth, gain in [(0, 0, 1), (90, 1, 20_000), (320, 1.6, 50_000)]:
    geometry = scene_geometry(phase, depth, gain)
    assert np.allclose(geometry['z'][1], solar_height(orbit_x, orbit_y, depth))
    assert np.allclose(geometry['z'][6], parent_height(geometry['x'][6], geometry['y'][6], phase, depth, gain))
    for index, (cx, cy, cz, radius) in zip([2, 3, 5], body_centers(phase, depth, gain)):
        xyz = np.array([geometry[axis][index] for axis in ('x', 'y', 'z')])
        offsets = xyz-np.array([cx, cy, cz])[:, None, None]
        assert np.allclose(np.sqrt(np.sum(offsets**2, axis=0)), radius)
        assert np.allclose(np.ptp(xyz, axis=(1, 2)), 2*radius)
        assert Z_RANGE[0] < xyz[2].min() < xyz[2].max() < Z_RANGE[1]
    assert all(z is None or Z_RANGE[0] < z <= 0 for z in geometry['z'][0])
assert np.all(grid_height([0, 0.5, 1.2], [0, 1, 0.2], 120, depth=0) == 0)
assert np.allclose(body_centers(360-1e-7), body_centers(360+1e-7), atol=1e-7)
assert not np.allclose(moon_position(0), moon_position(360))  # A year is not an integer lunar period.
lunar_cycle = 360*LUNAR_DAYS/YEAR_DAYS
assert np.allclose(np.array(moon_position(lunar_cycle))-earth_position(lunar_cycle), [MOON_ORBIT_RADIUS, 0])
print('Verified parent-surface orbits, moving self-wells, round spheres, and continuous lunar motion.')


## 2. The wireframe, bodies, and orbit guides

Gold, blue, and silver spheres use procedural colors without downloaded assets.
Earth's continents and Luna's craters are stylized, not geographic maps. Sizes and
the Earth–Luna separation are exaggerated independently for readability.

The blue ring follows the solar surface; the silver ring follows the Sun-and-Earth
surface. These are center trajectories. A body's own local dip lies below its
center, so the drawing does not force contact between a sphere and its own well.
Equal display scale on every axis keeps all three spheres round. Camera position
is retained as the geometry updates.


In [ ]:
# Simple stylized continents, drawn with overlapping smooth patches in longitude/latitude.
lon_deg, lat_deg = np.rad2deg(longitude), np.rad2deg(latitude)
land = np.zeros_like(longitude)
for lon, lat, width, height in [(-105, 42, 28, 24), (-62, -18, 17, 31),
                               (18, 2, 22, 29), (65, 44, 58, 23),
                               (135, -26, 19, 13), (-42, 72, 12, 13)]:
    delta_lon = (lon_deg - lon + 180) % 360 - 180
    land = np.maximum(land, np.exp(-((delta_lon/width)**2 + ((lat_deg-lat)/height)**2)))
earth_colors = np.where(land > 0.43, 0.61 + 0.08*np.cos(5*latitude),
                        0.18 + 0.06*np.sin(3*longitude)*np.cos(4*latitude))
earth_colors[np.abs(lat_deg) > 76] = 0.96
solar_colors = 0.55 + 0.16*np.sin(11*longitude)*np.cos(9*latitude) + 0.08*np.cos(23*latitude + longitude)

# Deterministic spherical crater patches (seamless at longitude +/-pi).
moon_colors = 0.60 + 0.06*np.cos(4*longitude)*np.cos(3*latitude)
for lon, lat, size in [(-1.4, 0.4, 0.24), (0.4, -0.3, 0.32), (2.1, 0.6, 0.20),
                       (-2.5, -0.5, 0.19), (1.3, 0.1, 0.16), (-0.3, 0.8, 0.21)]:
    angular_distance = np.arccos(np.clip(np.sin(latitude)*np.sin(lat)
        + np.cos(latitude)*np.cos(lat)*np.cos(longitude-lon), -1, 1))
    moon_colors -= 0.23*np.exp(-(angular_distance/size)**4)
    moon_colors += 0.12*np.exp(-((angular_distance-size)/(0.20*size))**2)

initial = scene_geometry()
fig = go.Figure([
    go.Scatter3d(x=initial['x'][0], y=initial['y'][0], z=initial['z'][0], mode="lines",
                 line=dict(color="#8fabbc", width=1.5), opacity=1.0,
                 hoverinfo="skip", connectgaps=False, name="Potential grid"),
    go.Scatter3d(x=initial['x'][1], y=initial['y'][1], z=initial['z'][1], mode="lines",
                 line=dict(color="#57bce6", width=3), opacity=0.85,
                 hoverinfo="skip", name="Circular orbit guide"),
    go.Surface(x=initial['x'][2], y=initial['y'][2], z=initial['z'][2],
               surfacecolor=solar_colors.tolist(), cmin=0, cmax=1,
               colorscale=[[0, '#be4b0d'], [0.35, '#f58b20'], [0.7, '#ffbe4b'], [1, '#fff1ae']],
               lighting=dict(ambient=0.85, diffuse=0.6, specular=0.15, roughness=0.7),
               showscale=False, name="Sun", hovertemplate="Sun<br>Mass: 1.9884 × 10³⁰ kg<extra></extra>"),
    go.Surface(x=initial['x'][3], y=initial['y'][3], z=initial['z'][3],
               surfacecolor=earth_colors.tolist(), cmin=0, cmax=1,
               colorscale=[[0, '#073477'], [0.35, '#2691d4'], [0.44, '#1b75af'],
                           [0.45, '#27674b'], [0.75, '#82b782'], [0.85, '#bad4d9'], [1, '#f1f9ff']],
               lighting=dict(ambient=0.55, diffuse=0.8, specular=0.3, roughness=0.6),
               lightposition=dict(x=0, y=0, z=2), showscale=False, name="Earth",
               hovertemplate="Earth<br>Mass: 5.9722 × 10²⁴ kg<br>Circular guide: 1 AU<extra></extra>"),
    go.Scatter3d(x=initial['x'][4], y=initial['y'][4], z=initial['z'][4], mode="text",
                 text=["SUN", "EARTH", "LUNA"], textfont=dict(color=["#ffd68c", "#8edbfa", "#d7deea"], size=11),
                 hoverinfo="skip", name="Labels"),
    go.Surface(x=initial['x'][5], y=initial['y'][5], z=initial['z'][5],
               surfacecolor=moon_colors.tolist(), cmin=0, cmax=1,
               colorscale=[[0, '#414752'], [0.4, '#858c96'], [0.7, '#c6cbd1'], [1, '#f5f2e9']],
               lighting=dict(ambient=0.65, diffuse=0.8, specular=0.1, roughness=1),
               lightposition=dict(x=0, y=0, z=2), showscale=False, name="Luna",
               hovertemplate="Luna · Moon<br>Mass: 7.346 × 10²² kg<br>Period: 27.3217 days<br>Separation and size enlarged<extra></extra>"),
    go.Scatter3d(x=initial['x'][6], y=initial['y'][6], z=initial['z'][6], mode="lines",
                 line=dict(color="#c4cedd", width=2), opacity=0.9,
                 hoverinfo="skip", name="Lunar orbit guide"),
])
fig.update_layout(
    template="none", autosize=True, showlegend=False,
    paper_bgcolor=BACKGROUND, plot_bgcolor=BACKGROUND,
    margin=dict(l=0, r=0, t=0, b=0), font=dict(family="system-ui, sans-serif", color="#e7f4fb"),
    hoverlabel=dict(bgcolor="#102331", bordercolor="#5f859d", font_color="#e7f4fb"),
    uirevision="gravity-camera", scene=dict(
        bgcolor=BACKGROUND, camera=CAMERA, dragmode="orbit", aspectmode="manual", aspectratio=SCENE_ASPECT,
        xaxis=dict(visible=False, range=[-EXTENT, EXTENT], autorange=False),
        yaxis=dict(visible=False, range=[-EXTENT, EXTENT], autorange=False),
        zaxis=dict(visible=False, range=list(Z_RANGE), autorange=False),
        uirevision="gravity-camera",
    ),
)

## 3. Browser geometry and controls

The exported page evaluates the same softened-potential formula locally as Earth and Luna move.
Elapsed phase is unbounded during playback so Luna does not jump at the year boundary.
The position slider selects a time in the first model year.
This avoids storing thousands of full grid frames and permits continuous control of depth.
The small JavaScript geometry function below is deliberately visible in the notebook.
The initial Python scene and browser scene are checked against one another during validation.

Playback uses elapsed time, at most 20 updates per second. Slow rendering coalesces updates
instead of accumulating stale frames. Orbit scrubbing pauses playback; camera reset changes
only the viewing angle. Reduced-motion preferences start paused, and hidden tabs suspend playback.

In [ ]:
model = dict(
    moonMassRatio=MOON_MASS_RATIO, lunarDays=LUNAR_DAYS,
    moonSoftening=MOON_SOFTENING, moonWellRadius=MOON_WELL_RADIUS, moonRadius=MOON_RADIUS,
    moonOrbitRadius=MOON_ORBIT_RADIUS, moonOffsetX=moon_offset_x.tolist(), moonOffsetY=moon_offset_y.tolist(),
    massRatio=MASS_RATIO, sunSoftening=SUN_SOFTENING, earthSoftening=EARTH_SOFTENING,
    sunWellRadius=SUN_WELL_RADIUS, earthWellRadius=EARTH_WELL_RADIUS,
    scale=POTENTIAL_SCALE, sunRadius=SUN_RADIUS, earthRadius=EARTH_RADIUS,
    orbitRadius=ORBIT_RADIUS, yearDays=YEAR_DAYS, secondsPerYear=SECONDS_PER_YEAR,
    initialPhase=INITIAL_PHASE, defaultDepth=DEFAULT_DEPTH, defaultGain=DEFAULT_GAIN,
    gridX=grid_x, gridY=grid_y, orbitX=orbit_x.tolist(), orbitY=orbit_y.tolist(),
    unitSphere=[part.tolist() for part in unit_sphere], camera=CAMERA,
)

geometry_script = r"""
const gravityModel = (() => {
  const model = __MODEL__;
  function localizedWell(distanceSquared, softening, displayRadius) {
    const weight = Math.max(0, 1-distanceSquared/(displayRadius*displayRadius))**3;
    return weight/Math.sqrt(distanceSquared+softening*softening);
  }
  function solarHeight(x,y,depth) {
    return -model.scale*depth*localizedWell(x*x+y*y,model.sunSoftening,model.sunWellRadius);
  }
  function parentHeight(x,y,centers,depth,gain) {
    const dx=x-centers[1][0], dy=y-centers[1][1];
    return solarHeight(x,y,depth)-model.scale*depth*model.massRatio*gain*
      localizedWell(dx*dx+dy*dy,model.earthSoftening,model.earthWellRadius);
  }
  function bodies(phase,depth=model.defaultDepth,gain=model.defaultGain) {
    const angle=phase*Math.PI/180, lunarAngle=phase*(model.yearDays/model.lunarDays)*Math.PI/180;
    const ex=model.orbitRadius*Math.cos(angle), ey=model.orbitRadius*Math.sin(angle);
    const centers=[[0,0,0,model.sunRadius], [ex,ey,solarHeight(ex,ey,depth),model.earthRadius]];
    const mx=ex+model.moonOrbitRadius*Math.cos(lunarAngle), my=ey+model.moonOrbitRadius*Math.sin(lunarAngle);
    centers.push([mx,my,parentHeight(mx,my,centers,depth,gain),model.moonRadius]);
    return centers;
  }
  function height(x,y,centers,depth,gain) {
    const dx=x-centers[2][0], dy=y-centers[2][1];
    return parentHeight(x,y,centers,depth,gain)-model.scale*depth*model.moonMassRatio*gain*
      localizedWell(dx*dx+dy*dy,model.moonSoftening,model.moonWellRadius);
  }
  function sphere(center) {
    return model.unitSphere.map((part,axis) => part.map(row => row.map(value => center[axis]+center[3]*value)));
  }
  function scene(phase,depth,gain) {
    const centers=bodies(phase,depth,gain), [sun,earth,moon]=centers;
    const gridZ=model.gridX.map((x,i) => x===null ? null : height(x,model.gridY[i],centers,depth,gain));
    const sunSphere=sphere(sun), earthSphere=sphere(earth), moonSphere=sphere(moon);
    const lunarX=model.moonOffsetX.map(x=>earth[0]+x), lunarY=model.moonOffsetY.map(y=>earth[1]+y);
    return {
      x:[model.gridX,model.orbitX,sunSphere[0],earthSphere[0],centers.map(c=>c[0]),moonSphere[0],lunarX],
      y:[model.gridY,model.orbitY,sunSphere[1],earthSphere[1],centers.map(c=>c[1]),moonSphere[1],lunarY],
      z:[gridZ,model.orbitX.map((x,i)=>solarHeight(x,model.orbitY[i],depth)),sunSphere[2],earthSphere[2],
        centers.map((c,i)=>c[2]+c[3]+[0.08,0.07,0.045][i]),moonSphere[2],
        lunarX.map((x,i)=>parentHeight(x,lunarY[i],centers,depth,gain))]
    };
  }
  return {model,scene,bodies,height,solarHeight,parentHeight,localizedWell};
})();
""".replace('__MODEL__', json.dumps(model, separators=(',', ':')))

controls_script = r"""
(function () {
  const gd = document.getElementById('{plot_id}');
  const {model, scene} = gravityModel;
  function setup() {
    const play = document.getElementById('play');
    const phaseSlider = document.getElementById('phase');
    const phaseValue = document.getElementById('phase-value');
    const depthSlider = document.getElementById('depth');
    const depthValue = document.getElementById('depth-value');
    const gainSelect = document.getElementById('earth-well');
    const speedSelect = document.getElementById('speed');
    const orbitToggle = document.getElementById('orbit');
    const reset = document.getElementById('reset-view');
    const status = document.getElementById('status');
    const modelNote = document.getElementById('model-note');
    const day = document.getElementById('day');
    const controls = [play, phaseSlider, depthSlider, gainSelect, speedSelect, orbitToggle, reset];
    const motion = window.matchMedia('(prefers-reduced-motion: reduce)');
    let playing = !motion.matches, phase = model.initialPhase;
    let depth = model.defaultDepth, gain = model.defaultGain;
    let raf = null, previous = null, lastDraw = -Infinity;
    let drawing = false, dirty = false, resetCamera = false, failed = false;

    function displayPhase() {
      const orbitalPhase = phase % 360;
      phaseSlider.value = String(orbitalPhase);
      phaseValue.textContent = Math.round(orbitalPhase) + '°';
      day.textContent = (phase % 360 / 360 * model.yearDays).toFixed(1);
    }
    function displaySettings() {
      play.textContent = playing ? 'Pause' : 'Play';
      play.setAttribute('aria-pressed', String(playing));
      status.textContent = playing ? 'Earth & Luna in motion' : 'Paused · drag to explore';
      depthValue.textContent = depth.toFixed(1) + '×';
      modelNote.textContent = gain === 1
        ? 'Real mass ratio · softened wells, heights and body sizes still illustrative.'
        : 'Earth & Luna wells amplified ×' + gain.toLocaleString('en-US') + ' · heights, sizes and lunar separation illustrative.';
    }
    function fail(error) {
      playing = false; failed = true;
      if (raf !== null) cancelAnimationFrame(raf);
      raf = null;
      displaySettings();
      status.textContent = 'The plot could not update. Reload the page to try again.';
      controls.forEach(control => { control.disabled = true; });
      console.error(error);
    }
    async function draw() {
      dirty = true;
      if (drawing || failed) return;
      drawing = true;
      try {
        while (dirty && !failed) {
          dirty = false;
          const geometry = scene(phase, depth, gain);
          if (resetCamera) {
            resetCamera = false;
            await Plotly.relayout(gd, {'scene.camera': JSON.parse(JSON.stringify(model.camera))});
          }
          await Plotly.restyle(gd, {
            ...geometry, visible: [true, orbitToggle.checked, true, true, true, true, orbitToggle.checked]
          }, [0, 1, 2, 3, 4, 5, 6]);
        }
      } catch (error) { fail(error); }
      finally { drawing = false; }
    }
    function schedule() {
      if (playing && !document.hidden && !failed && raf === null) raf = requestAnimationFrame(tick);
    }
    function tick(now) {
      raf = null;
      if (!playing || document.hidden || failed) return;
      if (previous !== null) phase += (now-previous)/1000 * 360/model.secondsPerYear * Number(speedSelect.value);
      previous = now;
      if (now-lastDraw >= 50) {
        lastDraw = now; displayPhase(); draw();
      }
      schedule();
    }
    function setPlaying(value) {
      playing = value; previous = null;
      if (raf !== null) cancelAnimationFrame(raf);
      raf = null; displaySettings(); schedule();
    }
    play.addEventListener('click', () => setPlaying(!playing));
    phaseSlider.addEventListener('input', () => {
      setPlaying(false); phase = Number(phaseSlider.value); displayPhase(); draw();
    });
    depthSlider.addEventListener('input', () => {
      depth = Number(depthSlider.value); displaySettings(); draw();
    });
    gainSelect.addEventListener('change', () => {
      gain = Number(gainSelect.value); displaySettings(); draw();
    });
    speedSelect.addEventListener('change', () => { previous = null; });
    orbitToggle.addEventListener('change', draw);
    reset.addEventListener('click', () => { resetCamera = true; draw(); });
    document.addEventListener('visibilitychange', () => {
      previous = null;
      if (raf !== null) cancelAnimationFrame(raf);
      raf = null; schedule();
    });
    motion.addEventListener('change', event => { if (event.matches) setPlaying(false); });
    controls.forEach(control => { control.disabled = false; });
    phaseSlider.value = String(phase); depthSlider.value = String(depth); gainSelect.value = String(gain);
    displayPhase(); displaySettings(); schedule();
  }
  if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', setup, {once: true});
  else setup();
})();
"""
interaction_script = geometry_script + '\n' + controls_script

## 4. Export the page

The layout and every plot-specific control are part of this cell. Only the local Plotly
dependency and legal footer come from the shared site helper. The export has a deterministic
plot ID and no generated timestamps, so fresh runs can be compared byte for byte.

In [ ]:
plot_html = fig.to_html(
    include_plotlyjs=False, full_html=False, div_id="gravity-plot",
    post_script=interaction_script.strip(), default_width="100%", default_height="100%",
    config=dict(responsive=True, displaylogo=False, displayModeBar=False, scrollZoom=True),
)

html_document = r"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <meta name="theme-color" content="#04080e">
  <title>Gravity / Sun, Earth &amp; Luna | phey.app</title>
  <style>
    * { box-sizing: border-box; }
    html, body { margin: 0; min-height: 100%; background: #04080e; color: #e7f4fb; }
    body { font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; }
    main { width: min(100%, 1240px); margin: auto; padding: 28px 32px 38px; }
    a { color: #9bd6ef; text-underline-offset: 4px; }
    .back { color: #a6b9c7; text-decoration: none; font-size: 13px; }
    header { display: flex; justify-content: space-between; align-items: end; gap: 24px; margin-top: 32px; }
    .eyebrow { color: #d5ae6d; font-size: 11px; letter-spacing: .18em; text-transform: uppercase; margin: 0 0 10px; }
    h1 { font-size: clamp(34px, 5vw, 54px); font-weight: 500; letter-spacing: -.045em; margin: 0; }
    h1 span { color: #8ba2b2; font-size: .48em; letter-spacing: -.02em; }
    .subtitle { color: #a6b9c7; font-size: 14px; line-height: 1.6; margin: 12px 0 0; max-width: 680px; }
    .legend { display: flex; gap: 20px; white-space: nowrap; color: #b6cbd7; font-size: 12px; padding-bottom: 5px; }
    .dot { display: inline-block; width: 8px; height: 8px; border-radius: 50%; margin-right: 7px; }
    .sun { background: #ffbd59; box-shadow: 0 0 14px #ffad4b66; }
    .earth { background: #6bc5ee; }
    .moon { background: #c4cedd; }
    .stage { position: relative; margin-top: 24px; border-top: 1px solid #21313e; border-bottom: 1px solid #21313e; }
    .plot { height: clamp(420px, 68vh, 730px); width: 100%; }
    .plot > div { height: 100%; }
    .stage-caption { position: absolute; top: 17px; left: 0; z-index: 1; pointer-events: none; color: #9ab3c3; font-size: 10px; letter-spacing: .09em; }
    .stage-hint { position: absolute; bottom: 14px; left: 0; z-index: 1; pointer-events: none; color: #91a7b6; font-size: 11px; }
    .timeline { display: flex; align-items: center; gap: 20px; padding: 20px 0; }
    button, select { font: inherit; font-size: 13px; min-height: 44px; border: 1px solid #354c5c; border-radius: 7px; background: #0c1822; color: #dceef8; padding: 10px 13px; cursor: pointer; }
    button:hover, select:hover { border-color: #a3ddf5; }
    button:disabled, select:disabled, input:disabled { opacity: .5; cursor: wait; }
    #play { min-width: 86px; background: #e9c78c; color: #1a1208; border-color: #e9c78c; font-weight: 650; }
    label { color: #b4c7d3; font-size: 12px; }
    .phase { flex: 1; display: grid; grid-template-columns: auto minmax(30px, 1fr) 40px; align-items: center; gap: 16px; }
    input[type=range] { width: 100%; min-width: 0; min-height: 44px; margin: 0; accent-color: #a3ddf5; cursor: pointer; }
    output { color: #dceef8; font-size: 12px; font-variant-numeric: tabular-nums; text-align: right; }
    .speed { display: flex; align-items: center; gap: 10px; }
    .settings { display: flex; flex-wrap: wrap; align-items: center; gap: 24px; padding: 0 0 20px; border-bottom: 1px solid #21313e; }
    .depth { flex: 1 1 250px; display: grid; grid-template-columns: auto minmax(70px, 1fr) 36px; align-items: center; gap: 14px; }
    .well { display: flex; gap: 12px; align-items: center; }
    .toggle { display: inline-flex; gap: 8px; align-items: center; min-height: 44px; cursor: pointer; }
    input[type=checkbox] { width: 17px; height: 17px; accent-color: #a3ddf5; }
    .readout { display: flex; justify-content: space-between; flex-wrap: wrap; gap: 12px; margin: 18px 0 8px; font-size: 12px; line-height: 1.6; color: #94acbd; }
    .readout p { margin: 0; }
    #status { color: #a3ddf5; }
    #day { color: #e7f4fb; font-variant-numeric: tabular-nums; }
    .model-note { margin: 0; font-size: 12px; color: #d5b783; line-height: 1.6; }
    details { margin-top: 25px; padding-top: 18px; border-top: 1px solid #21313e; color: #a6b9c7; font-size: 13px; line-height: 1.8; }
    summary { color: #dceef8; cursor: pointer; width: fit-content; min-height: 32px; }
    .explanation { max-width: 840px; }
    .sources { font-size: 12px; }
    :focus-visible { outline: 2px solid #a3ddf5; outline-offset: 4px; }
    @media (max-width: 700px) {
      main { padding: 20px 18px 28px; }
      header { display: block; margin-top: 26px; }
      .legend { margin-top: 18px; }
      .plot { height: 490px; height: min(120vw, 570px); min-height: 370px; }
      .timeline { flex-wrap: wrap; gap: 10px; }
      .phase { gap: 8px; min-width: 200px; }
      .speed { margin-left: auto; }
      .settings { gap: 12px 20px; }
      .depth { flex-basis: 100%; }
      .well { flex-wrap: wrap; }
      .readout { flex-direction: column; gap: 4px; }
    }
  </style>
</head>
<body>
  <main>
    <a class="back" href="../index.html">← phey.app</a>
    <header>
      <div>
        <p class="eyebrow">Physics in motion · 02</p>
        <h1>Gravity <span>/ Sun, Earth &amp; Luna</span></h1>
        <p class="subtitle">Earth follows the Sun’s well. Luna follows Earth, adding a smaller moving dip of its own.</p>
      </div>
      <div class="legend"><span><i class="dot sun" aria-hidden="true"></i>Sun</span><span><i class="dot earth" aria-hidden="true"></i>Earth</span><span><i class="dot moon" aria-hidden="true"></i>Luna</span></div>
    </header>
    <div class="stage">
      <span class="stage-caption">SPACETIME CURVATURE · SCHEMATIC VIEW</span>
      <div class="plot" role="region" aria-label="Sun, orbiting Earth and Luna with a deforming potential grid" aria-describedby="model-note plot-hint">__PLOT__</div>
      <span class="stage-hint" id="plot-hint">Drag to orbit the view · scroll to zoom</span>
    </div>
    <div class="timeline" role="group" aria-label="Playback controls">
      <button id="play" type="button" aria-pressed="false" disabled>Play</button>
      <div class="phase">
        <label for="phase">Year position</label>
        <input id="phase" type="range" min="0" max="360" step="0.1" value="320" title="Select a time in the first model year" disabled>
        <output id="phase-value" for="phase">320°</output>
      </div>
      <div class="speed">
        <label for="speed">Speed</label>
        <select id="speed" disabled><option value="0.25">0.25×</option><option value="0.5">0.5×</option><option value="1" selected>1×</option><option value="2">2×</option></select>
      </div>
    </div>
    <div class="settings" role="group" aria-label="Drawing controls">
      <div class="depth">
        <label for="depth">Grid depth</label>
        <input id="depth" type="range" min="0" max="1.6" step="0.1" value="1" disabled>
        <output id="depth-value" for="depth">1.0×</output>
      </div>
      <div class="well">
        <label for="earth-well">Earth &amp; Luna wells</label>
        <select id="earth-well" disabled>
          <option value="1">Real mass ratio</option>
          <option value="10000">Amplified ×10,000</option>
          <option value="20000" selected>Amplified ×20,000</option>
          <option value="50000">Amplified ×50,000</option>
        </select>
      </div>
      <label class="toggle"><input id="orbit" type="checkbox" checked disabled> Orbit guides</label>
      <button id="reset-view" type="button" disabled>Reset view</button>
    </div>
    <div class="readout">
      <p id="status" role="status">Loading the interactive plot…</p>
      <p>Day <span id="day">324.7</span> / 365.3 &nbsp;·&nbsp; Circular guide at 1 AU</p>
    </div>
    <p class="model-note" id="model-note">Earth &amp; Luna wells amplified ×20,000 · heights, sizes and lunar separation illustrative.</p>
    <noscript><p>Enable JavaScript to display this interactive plot.</p></noscript>
    <details>
      <summary>From a curved grid to curved spacetime</summary>
      <div class="explanation">
        <p>In general relativity, matter and energy shape spacetime, and freely falling test bodies follow its
        geodesics. This picture is a visual analogy: the grid uses localized, potential-shaped depressions to illustrate deformation. It is not a literal shape of spacetime, a time axis, or an exact general-relativistic solution.</p>
        <p>The Sun is roughly 333,000 times as massive as Earth. Choose <strong>Real mass ratio</strong> to see
        why Earth’s well is hard to distinguish on this scale. The amplified views apply the same gain to Earth and Luna’s contributions.
        All views retain enlarged body sizes and smoothed well centers. Grid depth changes the drawing only.</p>
        <p>Earth’s blue orbit and center follow the Sun-only surface. Luna’s silver orbit and center
        follow the combined Sun-and-Earth surface. Each body’s own well extends beneath it.
        The Sun stays at the reference height above its depression. This is a drawing convention:
        bodies are not physically rolling on a sheet, and gravity has no well edge. Our smooth
        display boundaries are not physical cutoffs.</p>
        <p>The circular guides have periods of 365.256 days for Earth and 27.3217 days for Luna,
        with one Earth year lasting 24 seconds at 1× speed. Luna’s orbit is enlarged for visibility;
        its actual mean distance from Earth is 384,400 km. Sizes and heights are illustrative.
        The orbits are prescribed, not calculated from the drawn wells. Orbital eccentricity,
        inclination, and Earth–Moon barycentric motion are omitted. Year position selects a
        time in the first model year; playback continues smoothly into subsequent years.</p>
        <p class="sources">Read more:
          <a href="https://www.einstein-online.info/en/GeomGravity/" target="_blank" rel="noopener noreferrer">Einstein Online · gravity and curvature</a>.
          Physical parameters:
          <a href="https://nssdc.gsfc.nasa.gov/planetary/factsheet/sunfact.html" target="_blank" rel="noopener noreferrer">NASA · Sun</a> /
          <a href="https://nssdc.gsfc.nasa.gov/planetary/factsheet/earthfact.html" target="_blank" rel="noopener noreferrer">Earth</a> /
          <a href="https://nssdc.gsfc.nasa.gov/planetary/factsheet/moonfact.html" target="_blank" rel="noopener noreferrer">Moon</a>.
        </p>
      </div>
    </details>
  </main>
</body>
</html>
""".replace('__PLOT__', plot_html)

output_dir = project_root / "output"
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "20261018_gravity.html"
write_plot_page(output_file, html_document)
print(f"Exported: {output_file.relative_to(project_root)}")

## 5. Interactive notebook preview

This iframe embeds the local Plotly bundle, so the preview works without a web server.
It contains the same plot, controls, and explanation as the exported page. Trust the
notebook and rerun this cell if your Jupyter environment suppresses interactive output.

In [ ]:
plotly_bundle = (project_root / PLOTLY_URL.lstrip("/")).read_text(encoding="utf-8")
preview_document = html_document.replace("</head>", "<script>" + plotly_bundle + "</script>\n</head>")
display(HTML(
    '<iframe title="Gravity / Sun, Earth and Luna" sandbox="allow-scripts allow-downloads allow-popups allow-popups-to-escape-sandbox" '
    'style="width:100%;height:1150px;border:0;background:#04080e" '
    f'srcdoc="{escape(preview_document, quote=True)}"></iframe>'
))